# syft-ingest API Exploration

## 1. Environment check

In [1]:
import os

token = os.getenv("BRIGHTDATA_API_TOKEN")
print(
    f"BRIGHTDATA_API_TOKEN : {token[:3] + '...' + token[-3:] if token else 'NOT SET -- BrightData cells will fail'}"
)

BRIGHTDATA_API_TOKEN : 7a5...1d5


## 2. Core imports

In [2]:
import syft_ingest as si
from syft_ingest import FetchConfig, gather

## 3. Simplified gather() API — YouTube video

In [3]:
# Simplest case: just platform + URLs
corpus = si.gather(
    "youtube", ["https://www.youtube.com/watch?v=kCc8FmEb1nY"]
)  # Karpathy: GPT from scratch

print(f"Items fetched: {len(corpus.all_items())}")
if corpus.all_items():
    item = corpus.all_items()[0]
    print(f"  Title: {item.title}")
    print(f"  Author: {item.author}")

2026-04-13 17:20:44.129 | DEBUG    | syft_ingest.sources.youtube:__init__:83 - YtDlpFetcher initialized with config: {'socket_timeout': 30, 'playlistend': 50, 'download_full_video': False}
2026-04-13 17:20:44.129 | INFO     | syft_ingest.core.registry:register_fetcher:77 - Registered fetcher YtDlpFetcher for youtube/yt-dlp
2026-04-13 17:20:44.130 | DEBUG    | syft_ingest.setup:register_fetchers:28 - Registered YtDlpFetcher for YouTube
2026-04-13 17:20:44.130 | INFO     | syft_ingest.core.registry:register_fetcher:77 - Registered fetcher BrightDataFetcher for facebook/brightdata
2026-04-13 17:20:44.130 | INFO     | syft_ingest.core.registry:register_fetcher:77 - Registered fetcher BrightDataFetcher for instagram/brightdata
2026-04-13 17:20:44.130 | DEBUG    | syft_ingest.setup:register_fetchers:37 - Registered BrightDataFetcher for Facebook and Instagram
2026-04-13 17:20:44.130 | INFO     | syft_ingest.core.registry:register_fetcher:77 - Registered fetcher LocalFetcher for local/local
2

Items fetched: 1
  Title: Let's build GPT: from scratch, in code, spelled out.
  Author: Andrej Karpathy


In [4]:
corpus.export("./corpus.jsonl")

2026-04-13 17:20:50.546 | INFO     | syft_ingest.core.exporters:_export_jsonl:74 - Exported 1 items to corpus.jsonl


## 4. Channel enumeration with config

In [5]:
# With config options passed as kwargs
corpus = si.gather(
    "youtube",
    ["https://www.youtube.com/@AndrejKarpathy"],
    playlistend=3,  # Cap at 5 videos to keep it fast
)

print(f"Videos found: {len(corpus.all_items())}")
for item in corpus.all_items():
    print(f"  [{item.published_at}] {item.title}")

2026-04-13 17:21:22.339 | INFO     | syft_ingest.sources.youtube:_fetch_async:150 - Fetching 1 YouTube URL(s)
2026-04-13 17:21:22.340 | INFO     | syft_ingest.sources.youtube:_fetch_async:161 - Detected channel/playlist URL: https://www.youtube.com/@AndrejKarpathy
2026-04-13 17:21:22.340 | DEBUG    | syft_ingest.sources.youtube:_enumerate_channel:340 - Enumerating channel with limit=3
2026-04-13 17:21:23.192 | INFO     | syft_ingest.sources.youtube:_enumerate_channel:377 - Enumerated 3 videos from channel
2026-04-13 17:21:23.193 | INFO     | syft_ingest.sources.youtube:_fetch_async:168 - Enumerated 3 videos from channel
2026-04-13 17:21:23.193 | DEBUG    | syft_ingest.sources.youtube:_extract_video_info_and_captions:430 - Extracting metadata for https://www.youtube.com/watch?v=EWvNQjAaOHw
2026-04-13 17:21:26.860 | DEBUG    | syft_ingest.sources.youtube:_extract_video_info_and_captions:523 - Extracted 3475 caption segments for en
2026-04-13 17:21:26.861 | DEBUG    | syft_ingest.sources.

Videos found: 3
  [2025-02-27 00:00:00+00:00] How I use LLMs
  [2025-02-05 00:00:00+00:00] Deep Dive into LLMs like ChatGPT
  [2024-06-09 00:00:00+00:00] Let's reproduce GPT-2 (124M)


In [6]:
corpus.export("corpus2.jsonl")

2026-04-13 17:21:38.944 | INFO     | syft_ingest.core.exporters:_export_jsonl:74 - Exported 3 items to corpus2.jsonl


## 5. Facebook scraping

Requires `BRIGHTDATA_API_TOKEN`. Triggers a real scrape job and polls until done (~60-120s).

In [ ]:
corpus = await gather(
    "facebook",
    ["https://www.facebook.com/karpathy"],
    author="Andrej Karpathy",
    timeout=120,
    poll_interval=5,
)

print(f"Items fetched: {len(corpus.all_items())}")
print(f"Author: {corpus.person}")

if corpus.all_items():
    item = corpus.all_items()[0]
    print("\nFirst item:")
    print(f"  Type: {type(item).__name__}")
    print(f"  Title: {item.title}")
    print(f"  Text: {(item.text or '')[:120]}")

## 6. Instagram scraping with posts_limit for testing

In [ ]:
corpus = await gather(
    "instagram",
    ["https://www.instagram.com/karpathy/"],
    author="Andrej Karpathy",
    timeout=120,
    poll_interval=5,
    posts_limit=5,  # Limit to 5 posts for quick testing (no posts_limit = all posts)
)

print(f"Items fetched: {len(corpus.all_items())}")
for i, item in enumerate(corpus.all_items()[:3], 1):
    print(f"\n{i}. {type(item).__name__}: {item.title}")

## 7. Error handling and validation

In [ ]:
# Invalid platform raises ValueError
try:
    corpus = gather("tiktok", ["https://tiktok.com/user"])
except ValueError as e:
    print(f"✅ ValueError caught for unsupported platform: {e}")

# Missing URLs raises ValueError
try:
    corpus = gather("youtube", None)
except ValueError as e:
    print(f"✅ ValueError caught for missing URLs: {e}")

## 8. Configuration validation with FetchConfig

In [ ]:
# FetchConfig provides validation and IDE autocomplete
# All these options are validated:

config = FetchConfig(
    # YouTube options
    socket_timeout=60,  # Network timeout (seconds)
    playlistend=10,  # Max videos from channel
    download_full_video=False,  # Enable full video download
    # BrightData options
    timeout=300,  # Scrape job timeout
    poll_interval=2,  # Check frequency
    posts_limit=5,  # Limit posts (for testing)
)

print("✅ FetchConfig created with validation:")
print(f"  socket_timeout: {config.socket_timeout}")
print(f"  posts_limit: {config.posts_limit}")
print()

# Invalid values are caught immediately
try:
    bad = FetchConfig(socket_timeout=-1)  # Must be >= 1
except Exception as e:
    print(f"✅ Validation error (expected): {type(e).__name__}")

In [ ]:
# Simplest: just platform + URLs
corpus = si.gather("youtube", ["https://youtube.com/watch?v=..."])

# With author
corpus = si.gather("youtube", ["https://youtube.com/watch?v=..."], author="Andrej")

# With config options
corpus = si.gather(
    "youtube", ["https://youtube.com/@user"], socket_timeout=60, playlistend=10
)

# With Instagram/Facebook (requires BRIGHTDATA_API_TOKEN)
corpus = si.gather(
    "instagram", ["https://instagram.com/user/"], timeout=120, posts_limit=5
)

# Export results
# corpus.export("jsonl", output="out.jsonl")

print("✅ API ready for use!")